In [1]:
import matplotlib.pyplot as plt
import numpy as np
import time
import scipy

from scipy import linalg
from scipy.sparse.linalg import eigsh
from scipy.sparse.linalg import spsolve
from scipy.sparse import diags
from scipy.sparse import identity

In [2]:
#GiBUU FinalEvents.dat data struct
# 1:Run  2:Event  3:ID 4:Charge     5:perweight   6:position(1)   7:position(2)   8:position(3)   9:momentum(0)  
#10:momentum(1)  11:momentum(2)  12:momentum(3)     13:history  14:production_ID  15:enu

# sscanf(tmpline.Data(),"%d %d %d %d %lf %lf %lf %lf %lf %lf %lf %lf %*d %d %lf", 
#& 1-tmprun, & 2-tmpevent, & 3-tmpid, & 4-tmpcharge, & 5-tmppw, & 6-tmppos1, & 7-tmppos2, & 8-tmppos3, 
#& 9-tmptote, & 10-tmpmom1, & 11-tmpmom2, & 12-tmpmom3, /*&history,*/ & 14-tmpprod, & 15-tmpenu);

    #1: nucleon (QE)
    #2-31: non-strange baryon resonance (as in IdTable)
    #32: pi neutron-background (e.g. nu + n -> mu + pi+ + n)
    #33: pi proton-background (e.g. nu + n -> mu + pi0 + p)
    #34: DIS
    #35: 2p2h QE
    #36: 2p2h Delta
    #37: two pion background


In [3]:
data_1_line = np.zeros(15)
data_1_line = [1, 4, 1, 1, 0.000000E+00, 0.000000E+00, 0.000000E+00, 0.000000E+00, 9.276043E-01, 1.171171E-01,-6.729271E-03, -1.334868E-01, 0, 35, 1.926118E+00]

run_n = data_1_line[0] #run
event_n = data_1_line[1] #event
particle_id = data_1_line[2] #id of particle
perweight = data_1_line[4] #weight of event
prod_id = data_1_line[13] #production id
beam_E = data_1_line[14] #energy of incoming neutrino from beam

part_pos = np.array([data_1_line[5], data_1_line[6], data_1_line[7]]) #position of particle in final state
part_4vec = np.array([data_1_line[8], data_1_line[9], data_1_line[10], data_1_line[11]])



In [4]:
part_4vec

array([ 0.9276043 ,  0.1171171 , -0.00672927, -0.1334868 ])

In [5]:
PionMass = 139.570/1e3


def read_event_csv(filename, run_n, event_n): #return event with the run number and event nunber
    filein = open(filename, "r")            # Open file for reading
    inputd = filein.readlines()
    inputdata = []
    temp_data_1_line = []

    leng = len(inputd)
    print("the number of line of the txt file list is : " + str(leng))

    for i in range(1,leng):
        tokens = inputd[i].split(",")
        
        for j in range(0,15):
            if j<4:
                tokens[j] = int(tokens[j])
            elif j<12:
                tokens[j] = float(tokens[j])
            elif j<14:
                tokens[j] = int(tokens[j])
            else:
                tokens[j] = float(tokens[j])
                
        temp_data_1_line = tokens

        if (temp_data_1_line[0] == float(run_n)) and (temp_data_1_line[1] == float(event_n)):
            inputdata.append(temp_data_1_line)
            
    del(inputd)            
    return inputdata

def read_event(filename, run_n, event_n): #return event with the run number and event nunber
    filein = open(filename, "r")            # Open file for reading
    inputd = filein.readlines()
    inputdata = []
    temp_data_1_line = []

    leng = len(inputd)
    print("the number of line of the txt file list is : " + str(leng))

    for i in range(1,leng):
        tokens = inputd[i].split()#("	")
        
        for j in range(0,15):
            if j<4:
                tokens[j] = int(tokens[j])
            elif j<12:
                tokens[j] = float(tokens[j])
            elif j<14:
                tokens[j] = int(tokens[j])
            else:
                tokens[j] = float(tokens[j])
                
        temp_data_1_line = tokens

        if (temp_data_1_line[0] == float(run_n)) and (temp_data_1_line[1] == float(event_n)):
            inputdata.append(temp_data_1_line)
            
        if (temp_data_1_line[0] >= float(run_n)) and (temp_data_1_line[1] > float(event_n)):
            break
         
    del(inputd)
    return inputdata

#4-vector is in form of [t,x,y,z]

def get_P(Lorentz_4vec):#return momentum from 4 momentum
    return np.linalg.norm(np.array([Lorentz_4vec[1],Lorentz_4vec[2],Lorentz_4vec[3]]))

def get_mag2(Lorentz_4vec):#return magnitude sq from 4 momentum
    return Lorentz_4vec[0]**2-(Lorentz_4vec[1]**2+Lorentz_4vec[2]**2+Lorentz_4vec[3]**2)

def get_mag(Lorentz_4vec):#return magnitude sq from 4 momentum
    return np.sqrt(Lorentz_4vec[0]**2-(Lorentz_4vec[1]**2+Lorentz_4vec[2]**2+Lorentz_4vec[3]**2))

def Dot_Lor(Lorentz_4vec1, Lorentz_4vec2):#return magnitude sq from 4 momentum
    return Lorentz_4vec1[0]*Lorentz_4vec2[0]-Lorentz_4vec1[1]*Lorentz_4vec2[1]-Lorentz_4vec1[2]*Lorentz_4vec2[2]-Lorentz_4vec1[3]*Lorentz_4vec2[3]

def get_theta(Lorentz_4vec):#return angle in degrees from z axis (0,0,1) from 4 momentum
    z_hat = np.array([0,0,1])
    p_vec = np.array([Lorentz_4vec[1],Lorentz_4vec[2],Lorentz_4vec[3]])
    p_vec = p_vec / np.linalg.norm(p_vec)
    angle_rad = np.arccos(np.dot(z_hat, p_vec))
    angle_deg = np.degrees(angle_rad)
    
    return angle_deg

#4-vector is in form of [t,x,y,z] instead of [x,y,z,t] for TLorentz vec for ROOT
def get_anal_event(event_lst):
    line_n = len(event_lst)
    print("num of FS particles in event : "+str(line_n))
    run_n, event_n, particle_id, perweight, prod_id, beam_E, part_posm, part_4vec, part_pos = [], [], [], [], [], [], [], [], []
    PU4pRecoil = [0,0,0,0]
    for i in range(0,line_n):
        run_n.append(event_lst[i][0]) #run
        event_n.append(event_lst[i][1]) #event
        particle_id.append(event_lst[i][2]) #id of particle
        perweight.append(event_lst[i][4]) #weight of event
        prod_id.append(event_lst[i][13]) #production id
        beam_E.append(event_lst[i][14]) #energy of incoming neutrino from beam

        part_pos.append(np.array([event_lst[i][5], event_lst[i][6], event_lst[i][7]])) #position of particle in final state
        part_4vec.append(np.array([event_lst[i][8], event_lst[i][9], event_lst[i][10], event_lst[i][11]]))
        if (particle_id[i] == 902) or (particle_id[i] == 901): #particle is muon get scatter should be changed to 901 for electron
            PU4pScatter = part_4vec[i]
            
        if perweight[i]<1e-10: #initial nucleon which should have 0 perweight
            iniNfullp = part_4vec[i]
            
        if (particle_id[i] == 1) and (get_P(part_4vec[i])>get_P(PU4pRecoil)):
            PU4pRecoil = part_4vec[i]
    
    beamE = sum(beam_E)/len(beam_E)
    print("beam E : "+str(beamE))
    beamMass = PionMass
    pz = np.sqrt(beamE*beamE-beamMass*beamMass)
    beamp = np.array([beamE,0,0,pz])
    dummyNu = np.array([beamE,0,0,beamE])
    
    print(" PU4pScatter : "+str(PU4pScatter))
    lvq= dummyNu - PU4pScatter
    
    scattermomentum = get_P(PU4pScatter)
    scattertheta = get_theta(PU4pScatter)
    
    recoilmomentum = get_P(PU4pRecoil)
    recoiltheta = get_theta(PU4pRecoil)
    
    Q2 = -get_mag2(lvq)
    print(" iniNfullp : "+str(iniNfullp))
    dummyW = lvq+iniNfullp
    print(" lvq : "+str(lvq))
    print(" dummyW : "+str(dummyW))
    Wtrue = get_mag(dummyW)

    xBj = Q2/(2*Dot_Lor(iniNfullp,lvq))
    #prod id
    #1: nucleon (QE)
    #2-31: non-strange baryon resonance (as in IdTable)
    #32: pi neutron-background (e.g. nu + n -> mu + pi+ + n)
    #33: pi proton-background (e.g. nu + n -> mu + pi0 + p)
    #34: DIS
    #35: 2p2h QE
    #36: 2p2h Delta
    #37: two pion background
    print()
    if prod_id[0] == 1:
        print("prod id : "+str(prod_id[0])+" event mode : QE")
    elif prod_id[0] <= 33:
        print("prod id : "+str(prod_id[0])+" event mode : RES")
    elif prod_id[0] == 34:
        print("prod id : "+str(prod_id[0])+" event mode : DIS")
    elif prod_id[0] == 35:
        print("prod id : "+str(prod_id[0])+" event mode : 2p2h QE")
        
    
    print(" run : "+str(run_n[0])+"\n event : "+str(event_n[0])+"\n prod_id : "+str(prod_id[0])+"\n beamE : "+str(beamE)+"\n scatter p : "+str(scattermomentum))
    print(" scattertheta : "+str(scattertheta)+"\n Q2 : "+str(Q2)+"\n Wtrue : "+str(Wtrue)+"\n xBj : "+str(xBj))
    print(" recoilmomentum : "+str(recoilmomentum)+"\n recoiltheta : "+str(recoiltheta))
    
    return [run_n[0], event_n[0], prod_id[0], beamE, scattermomentum, scattertheta, Q2, Wtrue, xBj]
    
    

In [6]:
filename = "FinalEvents.dat"
starttime = time.time()
event = read_event(filename,2,192)
endtime = time.time()
time_taken = endtime-starttime
print("file read in "+str(time_taken)+" secs")
print(event)

the number of line of the txt file list is : 3266107
file read in 1.381903886795044 secs
[[2, 192, 902, -1, 8.332335e-06, 0.0, 0.0, 0.0, 1.193904, -0.5676518, 0.0007299176, 1.044995, 0, 35, 1.264486], [2, 192, 1, 1, 0.0, 0.0, 0.0, 0.0, 0.923911, -0.209302, 0.04491779, -0.03547743, 0, 35, 1.264486], [2, 192, 1, 1, 8.332335e-06, 3.175275, 0.5515837, 0.9298329, 0.9391352, -0.1483814, -0.05768852, 0.04019523, 2001001, 35, 1.264486]]


In [7]:
event = read_event(filename,1,39)
print(event)
get_anal_event(event)

the number of line of the txt file list is : 3266107
[[1, 39, 902, -1, 1.34602e-05, 0.0, 0.0, 0.0, 5.078934, 0.1599039, 0.1339112, 5.07355, 0, 35, 5.258955], [1, 39, 1, 1, 0.0, 0.0, 0.0, 0.0, 0.8815418, -0.07877362, -0.0836699, 0.04271001, 0, 35, 5.258955], [1, 39, 1, 1, 1.34602e-05, -2.681968, -12.2533, 6.696898, 1.029851, -0.057789, -0.3721812, 0.1972089, 0, 35, 5.258955]]
num of FS particles in event : 3
beam E : 5.258955
 PU4pScatter : [5.078934  0.1599039 0.1339112 5.07355  ]
 iniNfullp : [ 0.8815418  -0.07877362 -0.0836699   0.04271001]
 lvq : [ 0.180021  -0.1599039 -0.1339112  0.185405 ]
 dummyW : [ 1.0615628  -0.23867752 -0.2175811   0.22811501]

prod id : 35 event mode : 2p2h QE
 run : 1
 event : 39
 prod_id : 35
 beamE : 5.258955
 scatter p : 5.077835273935208
 scattertheta : 2.3540634693387417
 Q2 : 0.0454689203046501
 Wtrue : 0.9851754295180017
 xBj : 0.17904414644782637
 recoilmomentum : 0.42514675630145643
 recoiltheta : 62.36348233631093


[1,
 39,
 35,
 5.258955,
 5.077835273935208,
 2.3540634693387417,
 0.0454689203046501,
 0.9851754295180017,
 0.17904414644782637]

In [15]:
event = read_event(filename,1,2)
print(event)
get_anal_event(event)

the number of line of the txt file list is : 3273614
[[1, 2, 902, -1, 3.810795e-05, 0.0, 0.0, 0.0, 0.2115095, 0.1303049, -0.07166957, 0.1070342, 0, 2, 1.583124], [1, 2, 1, 1, 0.0, 0.0, 0.0, 0.0, 0.9287117, 0.2737424, 0.04187007, 0.04686893, 0, 2, 1.583124], [1, 2, 1, 1, 3.810795e-05, -4.38312, -3.641817, 10.96292, 1.028272, -0.1213778, -0.1016902, 0.3904183, 5002001, 2, 1.583124], [1, 2, 1, 1, 3.810795e-05, 2.789166, 13.60771, 18.87359, 1.529036, 0.2017101, 0.6651524, 0.9874169, 1000002, 2, 1.583124], [1, 2, 1, 1, 3.810795e-05, -9.742529, 2.928312, 9.059379, 1.043236, -0.3059202, 0.1058818, 0.3220236, 3000002, 2, 1.583124], [1, 2, 1, 1, 3.810795e-05, -1.604262, 3.008959, -6.761292, 1.014934, -0.02769769, 0.1959832, -0.3332733, 10001001, 2, 1.583124], [1, 2, 1, 1, 3.810795e-05, 3.081829, -0.9875378, 2.085259, 0.9412236, 0.03127559, 0.118574, -0.02922888, 8001001, 2, 1.583124], [1, 2, 1, 0, 3.810795e-05, 7.002276, -5.860135, -8.120288, 1.069436, 0.3222885, -0.2037454, -0.3441894, 8000002

[1,
 2,
 2,
 1.583124,
 0.18322722013291284,
 54.25625791394709,
 0.31963065528178536,
 1.7142437886326571,
 0.1291623975735505]

In [11]:
event = read_event(filename,1,7)
print(event)
get_anal_event(event)

the number of line of the txt file list is : 3273614
[[1, 7, 902, -1, 0.0001199211, 0.0, 0.0, 0.0, 1.713522, -0.3092908, -0.7962609, 1.481655, 0, 32, 2.934033], [1, 7, 1, 0, 0.0, 0.0, 0.0, 0.0, 0.9216932, -0.1368567, 0.02512044, -0.1398624, 0, 32, 2.934033], [1, 7, 1, 0, 0.0001199211, 5.019498, 12.41462, 23.65765, 1.93878, 0.2306892, 0.8884108, 1.427071, 0, 32, 2.934033], [1, 7, 1, 1, 0.0001199211, 4.529215, 2.312442, -7.895184, 0.9914792, 0.07439722, 0.1341461, -0.2822354, -1000003, 32, 2.934033], [1, 7, 1, 1, 0.0001199211, -12.07014, 3.401872, 2.242618, 1.056885, -0.4544387, 0.1726135, 0.02917635, -1000003, 32, 2.934033]]
num of FS particles in event : 5
beam E : 2.934033
 PU4pScatter : [ 1.713522  -0.3092908 -0.7962609  1.481655 ]
 iniNfullp : [ 0.9216932  -0.1368567   0.02512044 -0.1398624 ]
 lvq : [1.220511  0.3092908 0.7962609 1.452378 ]
 dummyW : [2.1422042  0.1724341  0.82138134 1.3125156 ]

prod id : 32 event mode : RES
run : 1
 event : 7
 prod_id : 32
 beamE : 2.934033
 scatt

[1,
 7,
 32,
 2.934033,
 1.7102613130333182,
 29.964787873806248,
 1.34944697359645,
 1.470353974290298,
 0.49964866506467603]

In [12]:
event = read_event(filename,1,9)
print(event)
get_anal_event(event)

the number of line of the txt file list is : 3273614
[[1, 9, 902, -1, 2.155695e-05, 0.0, 0.0, 0.0, 1.460513, 0.6942136, 0.1694646, 1.269364, 0, 35, 2.015566], [1, 9, 1, 0, 0.0, 0.0, 0.0, 0.0, 0.9253907, -0.1666516, -0.1208753, -0.1469998, 0, 35, 2.015566], [1, 9, 1, 1, 2.155695e-05, -20.79973, 1.797657, 6.337797, 1.357972, -0.924075, 0.001741197, 0.3321525, 0, 35, 2.015566], [1, 9, 1, 0, 2.155695e-05, -0.07385226, -3.901583, 8.783381, 0.9898738, 0.05105289, -0.1541996, 0.2713343, 1001001, 35, 2.015566], [1, 9, 1, 1, 2.155695e-05, -3.326671, -3.627363, 2.354012, 0.9446447, -0.01928604, -0.1077759, 0.0250145, 1001001, 35, 2.015566]]
num of FS particles in event : 5
beam E : 2.015566
 PU4pScatter : [1.460513  0.6942136 0.1694646 1.269364 ]
 iniNfullp : [ 0.9253907 -0.1666516 -0.1208753 -0.1469998]
 lvq : [ 0.555053  -0.6942136 -0.1694646  0.746202 ]
 dummyW : [ 1.4804437 -0.8608652 -0.2903399  0.5992022]

prod id : 35 event mode : 2p2h QE
run : 1
 event : 9
 prod_id : 35
 beamE : 2.015566

[1,
 9,
 35,
 2.015566,
 1.4566865611977478,
 29.3776739129229,
 0.75938436507312,
 1.0036354528820715,
 0.7794048798913547]

In [13]:
event = read_event(filename,1,10)
print(event)
get_anal_event(event)

the number of line of the txt file list is : 3273614
[[1, 10, 902, -1, 3.731558e-05, 0.0, 0.0, 0.0, 3.274139, 0.3872948, 0.66661, 3.180323, 0, 35, 3.889101], [1, 10, 1, 0, 0.0, 0.0, 0.0, 0.0, 0.9182652, 0.2413459, -0.08285877, 0.01436102, 0, 35, 3.889101], [1, 10, 1, 0, 3.731558e-05, 10.78883, -13.96041, 8.334872, 1.231356, 0.4799252, -0.5325899, 0.3498766, 0, 35, 3.889101], [1, 10, 1, 1, 3.731558e-05, -17.45607, -9.042326, 4.314359, 1.204318, -0.6563098, -0.3260499, 0.1829923, 0, 35, 3.889101]]
num of FS particles in event : 4
beam E : 3.889101
 PU4pScatter : [3.274139  0.3872948 0.66661   3.180323 ]
 iniNfullp : [ 0.9182652   0.2413459  -0.08285877  0.01436102]
 lvq : [ 0.614962  -0.3872948 -0.66661    0.708778 ]
 dummyW : [ 1.5332272  -0.1459489  -0.74946877  0.72313902]

prod id : 35 event mode : 2p2h QE
run : 1
 event : 10
 prod_id : 35
 beamE : 3.889101
 scatter p : 3.2724334276706135
scattertheta : 13.626383422818114
 Q2 : 0.7185541460470399
 Wtrue : 1.1157289482471795
 xBj : 0.

[1,
 10,
 35,
 3.889101,
 3.2724334276706135,
 13.626383422818114,
 0.7185541460470399,
 1.1157289482471795,
 0.6061119472505501]

In [14]:
event = read_event(filename,1,12)
print(event)
get_anal_event(event)

the number of line of the txt file list is : 3273614
[[1, 12, 902, -1, 2.082902e-06, 0.0, 0.0, 0.0, 1.635429, 0.4392411, 0.8858935, 1.298355, 0, 35, 2.73725], [1, 12, 1, 1, 0.0, 0.0, 0.0, 0.0, 0.9258307, -0.07773526, -0.06011264, 0.173343, 0, 35, 2.73725], [1, 12, 1, 1, 2.082902e-06, -12.22493, -1.386091, 20.87007, 1.435684, -0.5849452, 0.02882401, 0.9156162, 0, 35, 2.73725], [1, 12, 1, 1, 2.082902e-06, 1.665179, -22.6386, 12.76764, 1.506493, 0.08345117, -1.037006, 0.5543749, 0, 35, 2.73725]]
num of FS particles in event : 4
beam E : 2.73725
 PU4pScatter : [1.635429  0.4392411 0.8858935 1.298355 ]
 iniNfullp : [ 0.9258307  -0.07773526 -0.06011264  0.173343  ]
 lvq : [ 1.101821  -0.4392411 -0.8858935  1.438895 ]
 dummyW : [ 2.0276517  -0.51697636 -0.94600614  1.612238  ]

prod id : 35 event mode : 2p2h QE
run : 1
 event : 12
 prod_id : 35
 beamE : 2.73725
 scatter p : 1.6320127889500313
scattertheta : 37.29234587934364
 Q2 : 1.8341493422554602
 Wtrue : 0.5914963010470476
 xBj : 1.342166

[1,
 12,
 35,
 2.73725,
 1.6320127889500313,
 37.29234587934364,
 1.8341493422554602,
 0.5914963010470476,
 1.342166400628616]

In [18]:
filename = "nu_mu_sample_jobcard4_MINERvA\GiBUUMINERvALE_nu_T0_Carbon_1_out1/FinalEvents.dat"
event = read_event(filename,2,12)
print(event)

the number of line of the txt file list is : 13669809
[[2, 12, 902, -1, 3.801258e-05, 0.0, 0.0, 0.0, 1.831453, -0.05037215, 0.2958759, 1.803601, 0, 35, 1.955155], [2, 12, 1, 1, 0.0, 0.0, 0.0, 0.0, 0.9158672, -0.1977203, 0.03576699, 0.03877834, 0, 35, 1.955155], [2, 12, 1, 1, 3.801258e-05, -1.192684, -0.3328881, 10.83487, 0.9791979, -0.02407384, 0.03010662, 0.2783857, 0, 35, 1.955155], [2, 12, 1, 1, 3.801258e-05, -2.558745, -0.8691897, -5.964559, 0.962879, -0.01214603, 0.06179227, -0.2081531, 0, 35, 1.955155]]


In [33]:
event = read_event(filename,7,8248)
print(event)
get_anal_event(event)

the number of line of the txt file list is : 13669809
[[7, 8248, 902, -1, 0.0001112534, 0.0, 0.0, 0.0, 2.529119, -0.1568405, -0.1737898, 2.516044, 0, 34, 5.151251], [7, 8248, 1, 1, 0.0, 0.0, 0.0, 0.0, 0.9085629, 0.06306799, -0.08375778, 0.01982305, 0, 34, 5.151251], [7, 8248, 1, 0, 0.0001112534, -6.993914, 1.597815, 15.53209, 1.14994, -0.2589687, 0.1286147, 0.5990922, 1001001, 34, 5.151251], [7, 8248, 101, 0, 0.0001112534, 19.66363, -12.18493, 13.18073, 0.3718823, 0.2789817, -0.1487294, 0.1389289, 2000010, 34, 5.151251], [7, 8248, 1, 1, 0.0001112534, 1.25765, 3.363306, 29.00245, 2.671542, 0.3086289, 0.3639089, 2.455527, 2000010, 34, 5.151251], [7, 8248, 1, 1, 0.0001112534, -5.092485, -3.863809, -9.417087, 0.9974436, -0.08951902, -0.1062925, -0.3094157, 2000002, 34, 5.151251], [7, 8248, 1, 1, 0.0001112534, 10.94556, -7.168531, 0.6983392, 1.069495, 0.4610607, -0.2202834, 0.05360076, 4000002, 34, 5.151251], [7, 8248, 1, 0, 0.0001112534, -1.378054, 9.203741, -6.309528, 1.110349, -0.0411863

[7,
 8248,
 34,
 5.151251000000001,
 2.5269110089119264,
 5.315613939720586,
 0.12354154444929133,
 2.3152011579738923,
 0.02645657359410647]

In [34]:
filename = "nu_e_sample_jobcard13_E7\GiBUUMINERvALE_nu_T0_Carbon_1_out1/FinalEvents.dat"
event = read_event(filename,2,12)
print(event)

the number of line of the txt file list is : 2520474
[[2, 12, 901, -1, 3.084846e-05, 0.0, 0.0, 0.0, 1.595742, -0.1367419, -0.2170863, 1.574981, 0, 35, 1.631925], [2, 12, 1, 0, 0.0, 0.0, 0.0, 0.0, 0.9097171, 0.06701041, -0.09384776, -0.1111457, 0, 35, 1.631925], [2, 12, 1, 1, 3.084846e-05, -2.579089, 0.586928, 0.02537319, 0.939303, -0.09090551, -0.1850331, 0.1323261, 0, 35, 1.631925]]


In [43]:
event = read_event(filename,2,11494)
print(event)
get_anal_event(event)

the number of line of the txt file list is : 2520474
[[2, 11494, 901, -1, 9.870628e-05, 0.0, 0.0, 0.0, 2.451184, -0.104074, 0.1793049, 2.442401, 0, 33, 3.85099], [2, 11494, 1, 1, 0.0, 0.0, 0.0, 0.0, 0.9183071, -0.06572914, 0.1151882, -0.1233749, 0, 33, 3.85099], [2, 11494, 1, 1, 9.870628e-05, 7.27128, -15.75614, 8.35934, 1.201066, 0.1894457, -0.6319405, 0.3570395, 0, 33, 3.85099], [2, 11494, 1, 1, 9.870628e-05, 2.215473, -4.322532, 0.44121, 0.9446193, -0.05081354, -0.1022997, -0.009800713, 4001001, 33, 3.85099], [2, 11494, 1, 1, 9.870628e-05, -0.3145993, 20.42611, 17.2977, 1.58053, 0.05141028, 0.9962378, 0.7893651, 1101001, 33, 3.85099], [2, 11494, 1, 1, 9.870628e-05, 5.255197, -3.568153, -0.8856054, 1.048029, 0.4381197, -0.1310421, -0.09696247, 6002001, 33, 3.85099], [2, 11494, 1, 0, 9.870628e-05, -0.7912965, -8.120287, -3.934768, 0.9963142, -0.01090502, -0.2665864, -0.2039869, 4001001, 33, 3.85099], [2, 11494, 1, 1, 9.870628e-05, -4.413761, 1.903372, 9.655876, 1.028291, -0.1349313, 0

[2,
 11494,
 33,
 3.85099,
 2.4511842626455094,
 4.8518498631406874,
 0.06764777792501042,
 1.9277686017895117,
 0.022750513097931677]